# **FactOrders ETL Pipeline**
## **Kimball Star Schema Fact Table**
- Reads transactional order data from **`orders_silver`**.
- Joins with dimensional tables (**`DimCustomersSCD2`**, **`DimProductsscd2`**, **`DimDate`**) using date-range matching for SCD2 to resolve the appropriate surrogate key versions.
- Loads facts and surrogate keys incrementally or initially to the gold zone.

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.window import Window
from pyspark.sql.types import *
from delta.tables import DeltaTable

### Initialize widgets and read configuration flags
- `init_load_flag == 1`: Initial load (table overwrite)
- `init_load_flag == 0`: Incremental load (merge updates matching `order_id`)

In [0]:
dbutils.widgets.text("init_load_flag", "")
init_load_flag = int(dbutils.widgets.get("init_load_flag"))

### Load source and dimension tables

In [0]:
df_orders = spark.read.table("databricks_cata.silver.orders_silver")
df_cust = spark.read.table("databricks_cata.gold.DimCustomersSCD2")
df_prod = spark.read.table("databricks_cata.gold.DimProductsscd2")
df_date = spark.read.table("databricks_cata.gold.DimDate")

In [0]:
df_orders.limit(5).display()

### Perform Surrogate Key Resolution (Kimball Date-Range Joins for SCD2 dimensions)

In [0]:
orders_alias = df_orders.alias("ord")
cust_alias = df_cust.alias("cust")
prod_alias = df_prod.alias("prod")
date_alias = df_date.alias("dt")

# SCD2 surrogate key matching: check if orders.order_date falls within dimension validity range
df_joined = orders_alias.join(
    cust_alias,
    (col("ord.customer_id") == col("cust.customer_id")) &
    (col("ord.order_date") >= col("cust.effective_start_date")) &
    ((col("ord.order_date") <= col("cust.effective_end_date")) | col("cust.effective_end_date").isNull()),
    "left"
).join(
    prod_alias,
    (col("ord.product_id") == col("prod.product_id")) &
    (col("ord.order_date") >= col("prod.effective_start_date")) &
    ((col("ord.order_date") <= col("prod.effective_end_date")) | col("prod.effective_end_date").isNull()),
    "left"
).join(
    date_alias,
    to_date(col("ord.order_date")) == col("dt.full_date"),
    "left"
)

### Select columns and handle missing/unknown dimension records
- Missing dimension matches are assigned a key of `-1` to preserve data integrity and prevent record drops.

In [0]:
df_fact = df_joined.select(
    coalesce(col("dt.DimDateKey"), date_format(col("ord.order_date"), "yyyyMMdd").cast("int")).alias("DimDateKey"),
    coalesce(col("cust.DimCustomerKey"), lit(-1)).alias("DimCustomerKey"),
    coalesce(col("prod.DimProductKey"), lit(-1)).alias("DimProductKey"),
    col("ord.order_id").alias("order_id"),
    col("ord.quantity").cast("int").alias("quantity"),
    col("ord.total_amount").cast("double").alias("total_amount"),
    current_timestamp().alias("create_date"),
    current_timestamp().alias("update_date")
)

In [0]:
df_fact.limit(10).display()

### Load/Merge data into the gold region Delta Lake Fact Table

In [0]:
if init_load_flag == 1:
    # Initial Load:
    df_fact.write.mode("overwrite")\
        .format("delta")\
        .option("path", "abfss://gold@customersprojectete.dfs.core.windows.net/FactOrders")\
        .saveAsTable("databricks_cata.gold.FactOrders")
else:
    # Incremental Load:
    if spark.catalog.tableExists("databricks_cata.gold.FactOrders"):
        dlt_obj = DeltaTable.forPath(spark, "abfss://gold@customersprojectete.dfs.core.windows.net/FactOrders")
        
        dlt_obj.alias("tgt").merge(
            df_fact.alias("src"),
            "tgt.order_id = src.order_id"
        ).whenMatchedUpdate(set={
            "DimDateKey": col("src.DimDateKey"),
            "DimCustomerKey": col("src.DimCustomerKey"),
            "DimProductKey": col("src.DimProductKey"),
            "quantity": col("src.quantity"),
            "total_amount": col("src.total_amount"),
            "update_date": current_timestamp()
        }).whenNotMatchedInsertAll().execute()
    else:
        # Fallback to overwrite if table does not exist yet
        df_fact.write.mode("overwrite")\
            .format("delta")\
            .option("path", "abfss://gold@customersprojectete.dfs.core.windows.net/FactOrders")\
            .saveAsTable("databricks_cata.gold.FactOrders")

In [0]:
%sql
select * from databricks_cata.gold.FactOrders order by order_id limit 10

In [0]:
%sql
select count(*) as total_rows from databricks_cata.gold.FactOrders